# Wavelet contrastive learning and its variants

The 50-frequency, 100-frequency, and normal-only experiments are configurations of one shared implementation. `trying.py`, `trying_20.py`, and the duplicated CNN definitions are replaced by the package and presets.

Install the project first with `pip install -e ".[neural,signal,wavelet,notebooks]"`. Opening this notebook does not start training or download data.


In [ ]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
print(f"Project: {ROOT.name}")


## Define the comparison

The two-class variants fit a supervised classifier on training embeddings. The normal-only variant uses paired cosine dissimilarity directly. Changing the frequency count also changes the CNN dense-layer input size, so record the resulting model and training budget when comparing runs.


In [ ]:
import pandas as pd
from ecg_afib.config import load_config

PRESETS = ["wavelet_contrastive_50", "wavelet_contrastive_100", "wavelet_normal_only"]
configs = [load_config(ROOT / "configs" / f"{name}.toml") for name in PRESETS]
pd.DataFrame([{"experiment": c.name, "method": c.method, "dataset": c.paths.dataset,
               "embedding_dim": c.model.embedding_dim, "epochs": c.training.epochs,
               "batch_size": c.training.batch_size} for c in configs])


## Run the shared training pipeline

GPU execution can be selected with the configuration device or CLI `--device cuda`. CWT inputs and CNN activations can use substantial memory; begin with the supplied small batch size.


In [ ]:
from ecg_afib.experiments import run_experiment

RUN_EXPERIMENTS = False
if RUN_EXPERIMENTS:
    for config in configs:
        run_experiment(config)
else:
    print("Wavelet training is disabled until the prepared data is available.")


## Compare completed runs

All scores increase toward AFib, including the normal-only `1 - cosine_similarity` score. Thresholds are chosen on validation records before test evaluation. No original notebook score is carried forward automatically.


In [ ]:
import json
from ecg_afib.evaluation import compare_runs
from ecg_afib.plots import plot_roc_curves

completed = [c.path("output") for c in configs if (c.path("output") / "run.json").exists()
             and json.loads((c.path("output") / "run.json").read_text())["status"] == "complete"]
if completed:
    comparison = pd.DataFrame(compare_runs(completed))
    figure, axis = plot_roc_curves(completed)
else:
    comparison = pd.DataFrame()
    print("No completed wavelet runs yet.")
comparison
